# Practical 07 — Game Playing - Minimax and Alpha-Beta Pruning

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 7 |
| **Related lectures** | Lectures 15 and 17 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent a two-player game as states, moves and terminal values.
2. Implement minimax and verify the resulting player cannot be beaten.
3. Add alpha-beta pruning and confirm it returns identical moves.
4. Measure the saving from pruning, and the effect of move ordering.
5. Add a depth limit and an evaluation function for a larger board.

## 2. Background theory

Lecture 15 showed that when an opponent chooses every second move, a plan is the wrong kind of answer — you need a **strategy** saying what to do in response to whatever they play. **Minimax** computes this by assuming the opponent always chooses the worst outcome for you: you maximise at your turn, they minimise at theirs, and values propagate up from the leaves.

Lecture 17 added **alpha-beta pruning**. Once one reply refutes a move, the remaining replies to it cannot change your decision, so they need never be examined. This is exact, not approximate: the move returned is identical, and only the work differs. How much is saved depends entirely on the order in which moves are considered.

**Key points to keep in mind while you work:**

- Minimax assumes a perfect opponent, so its guarantee is a floor rather than a target.
- Alpha-beta returns the same move as plain minimax; a different move means a bug.
- Good move ordering roughly doubles the depth reachable in the same time.
- Real games are too large to search to the end, so we stop early and estimate.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only.

Nothing to install.

In [1]:
print('Ready.')

Ready.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — The game: tic-tac-toe

A board is a tuple of nine cells. Using a tuple rather than a list means a state is immutable and can be used as a dictionary key later.

In [2]:
EMPTY = ' '
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def new_board():
    return tuple([EMPTY] * 9)

def moves(board):
    return [i for i, c in enumerate(board) if c == EMPTY]

def play(board, i, player):
    b = list(board); b[i] = player; return tuple(b)

def winner(board):
    for a, b, c in LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None

def terminal(board):
    return winner(board) is not None or not moves(board)

def show(board):
    for r in range(3):
        print(' ' + ' | '.join(board[3*r:3*r+3]))
        if r < 2: print(' ---+---+---')

show(play(new_board(), 4, 'X'))

   |   |  
 ---+---+---
   | X |  
 ---+---+---
   |   |  


### Step 2 — Minimax

X maximises and O minimises. The value is from X's point of view, so higher is better for X. Note how the two branches differ only in max versus min.

In [3]:
nodes_examined = 0

def value(board, player):
    """Minimax value of this board, from X's point of view."""
    global nodes_examined
    nodes_examined += 1
    w = winner(board)
    if w == 'X': return 1
    if w == 'O': return -1
    if not moves(board): return 0
    if player == 'X':
        return max(value(play(board, m, 'X'), 'O') for m in moves(board))
    else:
        return min(value(play(board, m, 'O'), 'X') for m in moves(board))

nodes_examined = 0
v = value(new_board(), 'X')
print('value of the empty board:', v, ' (0 = a draw with perfect play)')
print('nodes examined          :', nodes_examined)

value of the empty board: 0  (0 = a draw with perfect play)
nodes examined          : 549946


### Step 3 — Alpha-beta pruning

Two extra arguments carry the best values found so far for each player. When they cross, the remaining branches cannot matter and we stop early.

In [4]:
def value_ab(board, player, alpha=-2, beta=2):
    global nodes_examined
    nodes_examined += 1
    w = winner(board)
    if w == 'X': return 1
    if w == 'O': return -1
    if not moves(board): return 0
    if player == 'X':
        best = -2
        for m in moves(board):
            best = max(best, value_ab(play(board, m, 'X'), 'O', alpha, beta))
            alpha = max(alpha, best)
            if beta <= alpha:
                break                      # this branch is already refuted
        return best
    else:
        best = 2
        for m in moves(board):
            best = min(best, value_ab(play(board, m, 'O'), 'X', alpha, beta))
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best

nodes_examined = 0
v_plain = value(new_board(), 'X'); n_plain = nodes_examined
nodes_examined = 0
v_ab = value_ab(new_board(), 'X');  n_ab = nodes_examined

print('plain minimax : value %2d   nodes %6d' % (v_plain, n_plain))
print('alpha-beta    : value %2d   nodes %6d' % (v_ab, n_ab))
print('same value?   :', v_plain == v_ab)
print('saving        : %.0f%%' % (100 * (1 - n_ab / n_plain)))

plain minimax : value  0   nodes 549946
alpha-beta    : value  0   nodes  18297
same value?   : True
saving        : 97%


### Step 4 — A player that cannot be beaten

Choose the move with the best value. A correct minimax player never loses at tic-tac-toe, so any loss is a bug — usually maximise and minimise swapped.

In [5]:
def best_move(board, player):
    opts = moves(board)
    if player == 'X':
        return max(opts, key=lambda m: value_ab(play(board, m, 'X'), 'O'))
    return min(opts, key=lambda m: value_ab(play(board, m, 'O'), 'X'))

import random
random.seed(0)
losses = 0
for game in range(50):
    board, player = new_board(), 'X'
    while not terminal(board):
        if player == 'X':
            m = best_move(board, 'X')          # minimax player
        else:
            m = random.choice(moves(board))    # random opponent
        board = play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    if winner(board) == 'O':
        losses += 1
print('games played: 50   games lost by the minimax player:', losses)

games played: 50   games lost by the minimax player: 0


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Verify pruning changes nothing

For at least ten different board positions, confirm that plain minimax and alpha-beta return the same value and the same chosen move. Report any disagreement — it would indicate a bug.

> **Marks:** 4

In [6]:
# Task 1 - verify that pruning changes nothing
# Build 12 different positions by playing a few random moves from the empty board.
import random
random.seed(1)

def random_position(n_moves):
    """Play n_moves random moves (X first) and return (board, player_to_move)."""
    while True:
        board, player = new_board(), 'X'
        for _ in range(n_moves):
            board = play(board, random.choice(moves(board)), player)
            player = 'O' if player == 'X' else 'X'
            if terminal(board):
                break
        if not terminal(board):
            return board, player

def best_move_plain(board, player):
    """Same as best_move, but uses PLAIN minimax (value) instead of alpha-beta."""
    opts = moves(board)
    if player == 'X':
        return max(opts, key=lambda m: value(play(board, m, 'X'), 'O'))
    return min(opts, key=lambda m: value(play(board, m, 'O'), 'X'))

positions = [random_position(n) for n in [2, 3, 3, 4, 4, 4, 5, 5, 5, 5, 3, 4]]

disagreements = 0
print('%-3s %-6s %-12s %-12s %-10s %-10s' % ('#', 'turn', 'value plain', 'value a-b', 'move plain', 'move a-b'))
for k, (board, player) in enumerate(positions, 1):
    v1 = value(board, player)
    v2 = value_ab(board, player)
    m1 = best_move_plain(board, player)
    m2 = best_move(board, player)
    if v1 != v2 or m1 != m2:
        disagreements += 1
    print('%-3d %-6s %-12d %-12d %-10d %-10d' % (k, player, v1, v2, m1, m2))
print()
print('positions tested:', len(positions), '  disagreements:', disagreements)

#   turn   value plain  value a-b    move plain move a-b  
1   X      1            1            4          4         
2   O      1            1            0          0         
3   O      0            0            1          1         
4   X      1            1            4          4         
5   X      1            1            8          8         
6   X      1            1            6          6         
7   O      1            1            5          5         
8   O      1            1            2          2         
9   O      -1           -1           1          1         
10  O      0            0            6          6         
11  O      0            0            3          3         
12  X      1            1            0          0         

positions tested: 12   disagreements: 0


**Result of Task 1.** For all 12 positions, plain minimax and alpha-beta returned **the same value and the same chosen move** (0 disagreements). This is what we expect: alpha-beta only skips branches that cannot change the decision, so it can never return a different answer.

### Task 2 — Measure the effect of move ordering

Reverse the order in which moves are generated and count nodes examined again. Record both counts and the percentage difference.

> **Marks:** 4

In [7]:
# Task 2 - the effect of move ordering
# We redefine moves() with a switch, so the same search code can try the
# cells in normal order (0..8) or reversed order (8..0).
MOVE_ORDER = 'normal'

def moves(board):
    m = [i for i, c in enumerate(board) if c == EMPTY]
    return m[::-1] if MOVE_ORDER == 'reversed' else m

def count_ab(board, player, order):
    """Nodes examined by alpha-beta from this position, in the given move order."""
    global MOVE_ORDER, nodes_examined
    MOVE_ORDER = order
    nodes_examined = 0
    v = value_ab(board, player)
    n = nodes_examined
    MOVE_ORDER = 'normal'
    return v, n

# ---- (a) the empty board, as in the procedure
print('(a) the empty board')
for order in ['normal', 'reversed']:
    MOVE_ORDER = order
    mv = best_move(new_board(), 'X')
    MOVE_ORDER = 'normal'
    v, n = count_ab(new_board(), 'X', order)
    print('    %-9s order: alpha-beta %6d nodes | value %d | move chosen %d' % (order, n, v, mv))

# ---- (b) positions that are NOT symmetric under the 180-degree turn
print()
print('(b) other positions (X to move in the first one, the rest from Task 1)')
print('    %-22s %8s %9s %12s' % ('position', 'normal', 'reversed', 'difference'))
BOARD_B = (' ', ' ', 'X', 'X', ' ', ' ', ' ', 'O', 'O')
tests = [('X has 2 and 3, O has 7 and 8', BOARD_B, 'X')] + [('Task 1 position %d' % k, b, p) for k, (b, p) in enumerate(positions, 1)]
total_n = total_r = 0
for name, board, player in tests:
    v1, n1 = count_ab(board, player, 'normal')
    v2, n2 = count_ab(board, player, 'reversed')
    assert v1 == v2                      # ordering must never change the value
    total_n += n1; total_r += n2
    print('    %-22s %8d %9d %+11.0f%%' % (name, n1, n2, 100 * (n2 - n1) / n1))
print('    %-22s %8d %9d %+11.0f%%' % ('TOTAL', total_n, total_r, 100 * (total_r - total_n) / total_n))

(a) the empty board
    normal    order: alpha-beta  18297 nodes | value 0 | move chosen 0
    reversed  order: alpha-beta  18297 nodes | value 0 | move chosen 8

(b) other positions (X to move in the first one, the rest from Task 1)
    position                 normal  reversed   difference
    X has 2 and 3, O has 7 and 8       83        37         -55%
    Task 1 position 1           880       359         -59%
    Task 1 position 2           205       301         +47%
    Task 1 position 3           338       406         +20%
    Task 1 position 4            65        49         -25%
    Task 1 position 5            84        10         -88%
    Task 1 position 6            73        10         -86%
    Task 1 position 7            15        39        +160%
    Task 1 position 8            24        26          +8%
    Task 1 position 9            18        15         -17%
    Task 1 position 10           32        35          +9%
    Task 1 position 11          412       359       

**Result of Task 2 — read this carefully.**

*(a) On the empty board the node count does not change at all (18,297 normal, 18,297 reversed).* That is not a bug. Turning the board by 180° sends cell *i* to cell 8 − *i*, and tic-tac-toe looks the same after that turn, so trying the cells from 8 down to 0 explores exactly the mirror image of the tree. The only visible change is the move chosen: all nine first moves are worth 0 (a draw), so the first one tried wins the tie: cell 0 normally, cell 8 reversed. Both are equally good.

*(b) On positions that are not symmetric the order matters a lot.* For the position with X on 2 and 3 and O on 7 and 8, alpha-beta examines **83** nodes in normal order and **37** reversed, 55% fewer. Across the 13 positions the normal order examined 2,269 nodes and the reversed order 1,710, about 25% fewer in total, but it was better on some positions and worse on others (for example +160% on position 7). The value returned was **always the same** (the cell asserts it): order changes the work, never the answer.

### Task 3 — Play against your own program

Write a loop that lets you play against the minimax player. Record whether you ever managed better than a draw.

> **Marks:** 3

In [8]:
# Task 3 - play against your own program
def human_vs_minimax(human='X', get_move=input, verbose=True):
    """You play `human`; the minimax player plays the other side.
    get_move(prompt) must return the cell number you choose (0-8)."""
    computer = 'O' if human == 'X' else 'X'
    board, player = new_board(), 'X'
    while not terminal(board):
        if verbose:
            show(board); print()
        if player == human:
            while True:
                try:
                    m = int(get_move('Your move (0-8): '))
                    if m in moves(board):
                        break
                except ValueError:
                    pass
                print('That is not a free cell. Try again.')
        else:
            m = best_move(board, computer)
            if verbose:
                print('Minimax plays', m)
        board = play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    if verbose:
        show(board)
    w = winner(board)
    result = 'draw' if w is None else ('you win' if w == human else 'minimax wins')
    print('Result:', result)
    return result

# ---- To play for real, remove the # on the next line and run the cell:
# human_vs_minimax('X')

# ---- A cell cannot wait for typing in a saved notebook, so here is one game
# ---- played with a scripted list of moves (the "human" opens in the centre).
script = iter(['4', '8', '1', '3', '6'])
human_vs_minimax('X', get_move=lambda prompt: next(script))

   |   |  
 ---+---+---
   |   |  
 ---+---+---
   |   |  

   |   |  
 ---+---+---
   | X |  
 ---+---+---
   |   |  

Minimax plays 0
 O |   |  
 ---+---+---
   | X |  
 ---+---+---
   |   |  

 O |   |  
 ---+---+---
   | X |  
 ---+---+---
   |   | X

Minimax plays 2
 O |   | O
 ---+---+---
   | X |  
 ---+---+---
   |   | X

 O | X | O
 ---+---+---
   | X |  
 ---+---+---
   |   | X

Minimax plays 7
 O | X | O
 ---+---+---
   | X |  
 ---+---+---
   | O | X

 O | X | O
 ---+---+---
 X | X |  
 ---+---+---
   | O | X

Minimax plays 5
 O | X | O
 ---+---+---
 X | X | O
 ---+---+---
   | O | X

 O | X | O
 ---+---+---
 X | X | O
 ---+---+---
 X | O | X
Result: draw


'draw'

In [9]:
# Can ANY human ever do better than a draw? Check every possible human move.
from collections import Counter

def all_human_games(board, player, human):
    """Try every human move; the computer always answers with best_move."""
    if terminal(board):
        w = winner(board)
        return Counter({'draw' if w is None else ('human wins' if w == human else 'computer wins'): 1})
    total = Counter()
    if player == human:
        for m in moves(board):
            total += all_human_games(play(board, m, player), 'O' if player == 'X' else 'X', human)
    else:
        m = best_move(board, player)
        total += all_human_games(play(board, m, player), 'O' if player == 'X' else 'X', human)
    return total

for side in ['X', 'O']:
    print('human plays %s (all possible human games):' % side, dict(all_human_games(new_board(), 'X', side)))

human plays X (all possible human games): {'computer wins': 498, 'draw': 183}
human plays O (all possible human games): {'computer wins': 99, 'draw': 2}


**Result of Task 3.** I could not do better than a draw. The cell above plays one scripted game against the minimax player (I open in the centre and block every threat): it ends in a **draw**, which is the best I can get. The second cell goes further and checks **every possible human game**: playing X (moving first) there are 681 possible games and the computer wins 498 and draws 183, while the human wins **none**; playing O the human again wins **none** (computer wins 99, draws 2). So no human can beat this program. A draw is the best a human can ever get, and only by playing perfectly. (To play live, remove the `#` in front of `human_vs_minimax('X')`.)

### Task 4 — A larger board needs a depth limit

Extend the game to a 4 by 4 board needing four in a row. Exhaustive search is now infeasible, so add a depth limit and an evaluation function that scores a non-terminal position. Explain what your evaluation function rewards.

> **Marks:** 6

In [10]:
# Task 4 - a 4 x 4 board, four in a row, with a depth limit and an evaluation function
N4 = 4
LINES4 = ([tuple(range(r * N4, r * N4 + N4)) for r in range(N4)] +          # 4 rows
          [tuple(range(c, N4 * N4, N4)) for c in range(N4)] +               # 4 columns
          [tuple(i * (N4 + 1) for i in range(N4)),                          # main diagonal
           tuple((i + 1) * (N4 - 1) for i in range(N4))])                   # other diagonal

def new_board4():
    return tuple([EMPTY] * (N4 * N4))

def moves4(board):
    """Free cells, centre cells first (good ordering makes alpha-beta prune more)."""
    free = [i for i, c in enumerate(board) if c == EMPTY]
    centre = (N4 - 1) / 2
    return sorted(free, key=lambda i: abs(i // N4 - centre) + abs(i % N4 - centre))

def winner4(board):
    for line in LINES4:
        a = board[line[0]]
        if a != EMPTY and all(board[i] == a for i in line):
            return a
    return None

def terminal4(board):
    return winner4(board) is not None or EMPTY not in board

def evaluate4(board):
    """Score a NON-terminal position from X's point of view.
    For every line that still has no piece of the other player:
        a line holding k pieces of X is worth +10**(k-1)  (1, 10, 100)
        a line holding k pieces of O is worth -10**(k-1)
    Lines holding both X and O are dead and are worth 0."""
    score = 0
    for line in LINES4:
        cells = [board[i] for i in line]
        x, o = cells.count('X'), cells.count('O')
        if x and o:
            continue
        if x:
            score += 10 ** (x - 1)
        elif o:
            score -= 10 ** (o - 1)
    return score

nodes4 = 0

def value4(board, player, depth, alpha=-10**9, beta=10**9):
    """Depth-limited alpha-beta, value from X's point of view."""
    global nodes4
    nodes4 += 1
    w = winner4(board)
    if w == 'X': return 1000 + depth          # a quicker win scores higher
    if w == 'O': return -1000 - depth
    if EMPTY not in board: return 0
    if depth == 0:
        return evaluate4(board)               # stop searching and estimate
    if player == 'X':
        best = -10**9
        for m in moves4(board):
            best = max(best, value4(play(board, m, 'X'), 'O', depth - 1, alpha, beta))
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best
    best = 10**9
    for m in moves4(board):
        best = min(best, value4(play(board, m, 'O'), 'X', depth - 1, alpha, beta))
        beta = min(beta, best)
        if beta <= alpha:
            break
    return best

def best_move4(board, player, depth, rng=None):
    scored = [(value4(play(board, m, player), 'O' if player == 'X' else 'X', depth - 1), m)
              for m in moves4(board)]
    target = max(s for s, _ in scored) if player == 'X' else min(s for s, _ in scored)
    best = [m for s, m in scored if s == target]
    return rng.choice(best) if rng else best[0]

# ---- try it out
b = new_board4()
print('lines to check on a 4x4 board:', len(LINES4))
for d in [1, 2, 3, 4, 5]:
    nodes4 = 0
    mv = best_move4(b, 'X', d)
    print('depth %d: first move %2d, nodes examined %6d' % (d, mv, nodes4))

# a tiny position to check that evaluation and winning detection work
demo = ('X', 'X', 'X', ' ',
        'O', 'O', ' ', ' ',
        ' ', ' ', ' ', ' ',
        ' ', ' ', ' ', ' ')
print('\nDemo position (X to move; X needs cell 3 to win):')
for r in range(N4):
    print(' ' + ' | '.join(demo[N4*r:N4*r+N4]))
print('evaluation:', evaluate4(demo), '  depth-1 move for X:', best_move4(demo, 'X', 1))

lines to check on a 4x4 board: 10
depth 1: first move  5, nodes examined     16
depth 2: first move  5, nodes examined    256
depth 3: first move  5, nodes examined   1094
depth 4: first move  5, nodes examined   6444
depth 5: first move  5, nodes examined  34895

Demo position (X to move; X needs cell 3 to win):
 X | X | X |  
 O | O |   |  
   |   |   |  
   |   |   |  
evaluation: 91   depth-1 move for X: 3


**What my evaluation function rewards.** `evaluate4` looks at each of the 10 lines (4 rows, 4 columns, 2 diagonals). A line that still holds pieces of only one player is *alive* for that player and is worth 1, 10 or 100 points for 1, 2 or 3 pieces (positive for X, negative for O). A line that holds both X and O is *dead* and is worth 0. So it rewards **lines that are still possible, and the closer they are to four in a row the more they are worth**, and it punishes the same for the opponent. A real win or loss is scored ±1000 (plus the remaining depth, so quicker wins are preferred). I chose these numbers myself (the factor 10 and the ±1000), so the player's "taste" comes from me, not from the game. Cells are also tried centre-first so alpha-beta prunes more. The depth table above shows why a limit is needed: the work grows about five-fold with each extra depth (256, 1,094, 6,444, 34,895 nodes).

### Task 5 — Depth against strength

Play your depth-limited player at depth 2 against the same player at depth 4, over 20 games. Record the results and relate them to Lecture 17.

> **Marks:** 4

In [11]:
# Task 5 - depth 2 against depth 4 on the 4 x 4 board, 20 games
import time

def play_game4(depth_x, depth_o, rng):
    board, player = new_board4(), 'X'
    first = True
    while not terminal4(board):
        depth = depth_x if player == 'X' else depth_o
        if first:
            m = rng.choice(moves4(board))          # random opening, so games differ
            first = False
        else:
            m = best_move4(board, player, depth, rng)
        board = play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    return winner4(board)

rng = random.Random(0)
tally = {'depth 2 wins': 0, 'depth 4 wins': 0, 'draws': 0}
start = time.time()
for game in range(20):
    if game % 2 == 0:                  # depth 2 plays X in even games, O in odd games
        w = play_game4(2, 4, rng); d2 = 'X'
    else:
        w = play_game4(4, 2, rng); d2 = 'O'
    if w is None:
        tally['draws'] += 1
    elif w == d2:
        tally['depth 2 wins'] += 1
    else:
        tally['depth 4 wins'] += 1
print(tally)
print('time: %.1f seconds' % (time.time() - start))

{'depth 2 wins': 0, 'depth 4 wins': 0, 'draws': 20}
time: 2.8 seconds


**Result of Task 5.** Over 20 games (depth 2 played X in 10 games and O in 10, with a random first move and random choice between equally good moves) the result was **0 wins for depth 2, 0 wins for depth 4 and 20 draws**.

I did not expect that, but it makes sense. Four in a row on a 4×4 board is a game that is a draw when both sides block sensibly, and depth 2 already sees everything needed to block: its own immediate win and the opponent's immediate win. Depth 4 sees two moves further, but there is nothing extra to gain in this small game. (I also tried other random openings and depth 1 against depth 4 and still got draws.) Lecture 17's point still holds: deeper search can only be better or equal when the evaluation is sensible, but a stronger player does not have to win; against a player that already defends well, the result is a draw. The price of the extra depth is real: about 25 times more nodes per move (6,444 against 256 on the empty board).

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Method | Nodes examined | Value returned | Move chosen |
|---|---|---|---|
| Plain minimax | 549,946 | 0 | 0 |
| Alpha-beta (normal order) | 18,297 | 0 | 0 |
| Alpha-beta (reversed order) | 18,297 | 0 | 8 |

*(Empty board, X to move. Alpha-beta saves 96.7%. The reversed order gives the same count here only because of the board's 180° symmetry; on the position X on 2 and 3, O on 7 and 8 it was 83 nodes normal against 37 reversed. Both moves, 0 and 8, are draws, so each is a correct choice.)*

**Depth against strength (Task 5)**

| Depth 2 wins | Depth 4 wins | Draws |
|---|---|---|
| 0 | 0 | 20 |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Alpha-beta examined far fewer nodes and returned the same move. Explain in two sentences why that is guaranteed rather than lucky.
2. Reversing the move order changed the saving substantially. Why does the order matter so much?
3. Your depth-4 player beat your depth-2 player. Did alpha-beta make it play better, or did it do something else? Be precise.

*Your answers:*

1. Alpha-beta only skips a branch when one reply has already shown that the player to move above would never choose it (for example, MIN has found a reply that holds X to a value no better than what X already has elsewhere). Whatever the skipped replies are worth, they cannot change the value that reaches the top, so the answer is guaranteed to be identical; only the amount of work is smaller (18,297 against 549,946 nodes).

2. A cut only happens after a good reply has been found. If the best reply is tried first, the rest of the replies can be skipped; if it is tried last, they have all been examined already. So the same tree can cost very little or a lot depending on the order. Here, on the position with X on 2 and 3 and O on 7 and 8, the reversed order examined 37 nodes against 83. (On the empty board there was no change, because reversing the order just mirrors the board; the order effect shows up on non-symmetric positions.)

3. Alpha-beta did **not** make anything play better: at the same depth it returns exactly the same move as plain minimax. What it did was make the work cheaper, which is what lets us afford a bigger depth. Depth is what changes the strength, because the player sees more moves ahead. In my 4×4 experiment, however, depth 4 did not beat depth 2: all 20 games were draws, because depth 2 already sees the immediate wins and blocks, and four in a row on 4×4 is a draw with sensible play. So deeper search is not a guarantee of winning.

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Why does the opponent minimise rather than maximise?
2. Show me the line where a branch is cut, and explain what has just been established.
3. Your pruned search returned a different move from plain minimax. What would that mean?
4. Why does move ordering affect the saving but never the answer?
5. What does your evaluation function reward, and who decided that?
6. Minimax never loses at tic-tac-toe. Does it always win? Why not?

## Viva questions — short answers

1. Because the opponent wants us to lose. A win for us is a loss for them, so on their turn they pick the move that gives us the lowest value.
2. `if beta <= alpha: break`. It means the player above already has a better option than anything this branch can give, so no remaining reply in this branch can change the decision.
3. That there is a bug. Alpha-beta must return the same value and move as plain minimax.
4. A different order only changes how soon a refuting reply is found, so more or fewer branches are skipped. A skipped branch could never have changed the value, so the answer is the same.
5. It rewards lines that are still open for a player, more if they have more pieces in them (1, 10, 100 points). I decided that, not the game, so another programmer could pick different weights and get a different player.
6. No. It never loses, but it only wins if the opponent makes a mistake. With perfect play on both sides tic-tac-toe is a draw (the value of the empty board is 0).

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P07_<your-roll-number>.ipynb`

In [12]:
NAME = "MUHAMMAD HASHAIR"
ROLL_NUMBER = "2025-cse-048"
print(NAME, ROLL_NUMBER)

MUHAMMAD HASHAIR 2025-cse-048
